# 手写数字识别的神经网络，多类别

在本练习中，你将使用神经网络识别手写数字 0-9。


# 目录
- [ 1 - 程序包 ](#1)
- [ 2 - ReLU 激活函数](#2)
- [ 3 - Softmax 函数](#3)
  - [ 练习 1](#ex01)
- [ 4 - 神经网络](#4)
  - [ 4.1 问题陈述](#4.1)
  - [ 4.2 数据集](#4.2)
  - [ 4.3 模型表示](#4.3)
  - [ 4.4 Tensorflow 模型实现](#4.4)
  - [ 4.5 Softmax 的位置](#4.5)
    - [ 练习 2](#ex02)


<a name="1"></a>
## 1 - 程序包

首先，让我们运行下面的单元格，导入本作业需要用到的所有程序包。
- [numpy](https://numpy.org/) 是 Python 科学计算的基础程序包。
- [matplotlib](http://matplotlib.org) 是 Python 中常用的绘图库。
- [tensorflow](https://www.tensorflow.org/) 是流行的机器学习平台。

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.activations import linear, relu, sigmoid
%matplotlib widget
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')

import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

from public_tests import * 

from autils import *
from lab_utils_softmax import plt_softmax
np.set_printoptions(precision=2)

<a name="2"></a>
## 2 - ReLU 激活函数
本周介绍了一个新的激活函数：修正线性单元（ReLU）。
$$ a = max(0,z) \\quad\\quad\\text {# ReLU function} $$

In [ ]:
plt_act_trio()

<img align="right" src="./图片/C2_W2_ReLu.png"     style=" width:380px; padding: 10px 20px; " >

> **图中文字（中文）**：需求预测示例 输入特征：价格（price）、运输成本（shipping cost）、营销（marketing）、材料（material） 中间概念：可负担性（affordability）、认知度（awareness）、感知质量（perceived quality） sigmoid：0 < g(z) < 1 ReLU：g(z) = max(0, z) 线性（linear）：g(z) = z
右边课上举的例子展示了 ReLU 的一种应用。在这个例子中，推导出的“awareness（意识/警觉）”特征不是二元的，而是具有连续的取值范围。sigmoid 最适合开/关或二元情形。ReLU 提供了连续的线性关系。此外它还有一个输出为零的“关闭”区间。    
“关闭”特性使 ReLU 成为非线性激活函数。为什么需要这样？这使多个单元能够共同作用于最终函数而互不干扰。配套的可选实验对此有更深入的探讨。


<a name="3"></a>
## 3 - Softmax 函数
多类别神经网络生成 N 个输出，其中被选中作为预测答案的那个输出。在输出层，由一个线性函数生成向量 $\\mathbf{z}$，再把它送入 softmax 函数。softmax 函数按下面所述把 $\\mathbf{z}$ 转换为概率分布。应用 softmax 之后，每个输出都在 0 和 1 之间，且各输出之和为 1，可以解释为概率。送入 softmax 的输入越大，对应的输出概率越大。
<center>  <img  src="./图片/C2_W2_NNSoftmax.PNG" width="600" />

> **图中文字（中文）**：带 Softmax 输出的神经网络 Softmax 函数：a1 = P(y = 1|x) = …，…，a10 = P(y = 10|x) = … 输出 10 个单元  

softmax 函数可以写作：
$$a_j = \\frac{e^{z_j}}{ \\sum_{k=0}^{N-1}{e^{z_k} }} \\tag{1}$$

其中 $z = \\mathbf{w} \\cdot \\mathbf{x} + b$，N 是输出层中特征/类别的数量。  

<a name="ex01"></a>
### 练习 1
让我们写一个 NumPy 实现：

In [ ]:
# UNQ_C1
# 计分单元：my_softmax

def my_softmax(z):  
    """Softmax 把一组数值转换为概率分布。
    参数：
      z (ndarray (N,))  : 输入数据，N 个特征
    返回：
      a (ndarray (N,))  : z 的 softmax"""    
    ### 在此处编写代码 ###

    ### 代码编写结束 ###
    return a

In [ ]:
z = np.array([1., 2., 3., 4.])
a = my_softmax(z)
atf = tf.nn.softmax(z)
print(f"my_softmax(z):         {a}")
print(f"tensorflow softmax(z): {atf}")

# 单元测试开始
test_my_softmax(my_softmax)
# 单元测试结束

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    一种实现方式是用一个 for 循环先构建分母，再用第二个循环计算每个输出。
    

```python
def my_softmax(z):  
    N = len(z)
    a =                     # initialize a to zeros 
    ez_sum =                # initialize sum to zero
    for k in range(N):      # loop over number of outputs             
        ez_sum +=           # sum exp(z[k]) to build the shared denominator      
    for j in range(N):      # loop over number of outputs again                
        a[j] =              # divide each the exp of each output by the denominator   
    return(a)
```

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看代码</b></font></summary>
   

```python
def my_softmax(z):  
    N = len(z)
    a = np.zeros(N)
    ez_sum = 0
    for k in range(N):                
        ez_sum += np.exp(z[k])       
    for j in range(N):                
        a[j] = np.exp(z[j])/ez_sum   
    return(a)

Or, a vector implementation:

def my_softmax(z):  
    ez = np.exp(z)              
    a = ez/np.sum(ez)           
    return(a)

```



下面，改变 `z` 各输入的值。特别注意分子中的指数如何放大小数值之间的差异。同时注意各输出值之和为 1。

In [ ]:
plt.close("all")
plt_softmax(my_softmax)

<a name="4"></a>
## 4 - 神经网络

在上周的作业中，你实现了一个用于二元分类的神经网络。本周你将把它扩展到多类别分类，这会用到 softmax 激活函数。


<a name="4.1"></a>
### 4.1 问题陈述

在本练习中，你将使用神经网络识别十个手写数字 0-9。这是一个多类别分类任务，即从 n 个选项中选出一个。如今，自动手写数字识别被广泛使用——从识别信封上的邮政编码到识别银行支票上写的金额。


<a name="4.2"></a>
### 4.2 数据集

你将先加载本任务的数据集。
- 下面所示的 `load_data()` 函数把数据加载到变量 `X` 和 `y` 中


- 数据集包含 5000 个手写数字的训练样本$^1$。

    - 每个训练样本是一张 20 像素 × 20 像素的数字灰度图。
        - 每个像素用一个浮点数表示，代表该位置的灰度强度。
        - 20 × 20 的像素网格被“展开”成一个 400 维向量。
        - 每个训练样本成为数据矩阵 `X` 中的一行。
        - 这样我们就得到一个 5000 × 400 的矩阵 `X`，其中每一行是一张手写数字图像的训练样本。

$$X = 
\left(\begin{array}{cc} 
--- (x^{(1)}) --- \\
--- (x^{(2)}) --- \\
\vdots \\ 
--- (x^{(m)}) --- 
\end{array}\right)$$ 

- 训练集的第二部分是一个 5000 × 1 维向量 `y`，包含训练集的标签
    - 如果图像是数字 `0`，则 `y = 0`；如果图像是数字 `4`，则 `y = 4`，依此类推。

$^1$<sub> 这是 MNIST 手写数字数据集的一个子集（http://yann.lecun.com/exdb/mnist/）</sub>

In [ ]:
# 加载数据集
X, y = load_data()

#### 4.2.1 查看变量
让我们更熟悉一下你的数据集。  
- 一个好的起点是把每个变量打印出来，看看它包含什么。

下面的代码打印变量 `X` 和 `y` 中的第一个元素。  

In [ ]:
print ('The first element of X is: ', X[0])

In [ ]:
print ('The first element of y is: ', y[0,0])
print ('The last element of y is: ', y[-1,0])

#### 4.2.2 检查变量的维度

熟悉数据的另一种方式是查看它的维度。请打印 `X` 和 `y` 的形状，看看数据集中有多少训练样本。

In [ ]:
print ('The shape of X is: ' + str(X.shape))
print ('The shape of y is: ' + str(y.shape))

#### 4.2.3 可视化数据

你将先可视化训练集的一个子集。
- 在下面的单元格中，代码从 `X` 中随机选取 64 行，把每一行映射回 20 像素 × 20 像素的灰度图，并把图像一起显示出来。
- 每张图像的标签显示在图像上方

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# 你不需要修改本单元格中的任何内容

m, n = X.shape

fig, axes = plt.subplots(8,8, figsize=(5,5))
fig.tight_layout(pad=0.13,rect=[0, 0.03, 1, 0.91]) #[左, 下, 右, 上]

#fig.tight_layout(pad=0.5)
widgvis(fig)
for i,ax in enumerate(axes.flat):
    # 随机选取索引
    random_index = np.random.randint(m)
    
    # 选取与随机索引对应的行，并
    # 重塑图像
    X_random_reshaped = X[random_index].reshape((20,20)).T
    
    # 显示图像
    ax.imshow(X_random_reshaped, cmap='gray')
    
    # 在图像上方显示标签
    ax.set_title(y[random_index,0])
    ax.set_axis_off()
    fig.suptitle("Label, image", fontsize=14)

<a name="4.3"></a>
### 4.3 模型表示

本作业中你将使用的神经网络如下图所示。
- 它包含两个带 ReLU 激活函数的密集层，后面接一个使用线性激活的输出层。
    - 回忆一下，我们的输入是数字图像的像素值。
    - 由于图像尺寸为 $20\times20$，因此我们有 $400$ 个输入  
    
<img src="图片/C2_W2_Assigment_NN.png" width="600" height="450">

> **图中文字（中文）**：网络结构：输入（400）→ 25 个单元 → 15 个单元 → 10 个单元 → 输出概率 （softmax 单独应用，见下文）

- 这些参数的维度按这样一个神经网络来确定：第 1 层 $25$ 个单元，第 2 层 $15$ 个单元，第 3 层 $10$ 个输出单元，每个数字一个。

    - 回忆一下，这些参数的维度按如下方式确定：
        - 如果网络某一层有 $s_{in}$ 个单元、下一层有 $s_{out}$ 个单元，那么
            - $W$ 的维度为 $s_{in} \\times s_{out}$。
            - $b$ 是一个含 $s_{out}$ 个元素的向量
  
    - 因此，`W` 和 `b` 的形状分别为
        - 第 1 层：`W1` 的形状是 (400, 25)，`b1` 的形状是 (25,)
        - 第 2 层：`W2` 的形状是 (25, 15)，`b2` 的形状是 (15,)
        - 第 3 层：`W3` 的形状是 (15, 10)，`b3` 的形状是 (10,)
>**注意：** 偏置向量 `b` 可以表示为 1 维数组 (n,) 或 2 维数组 (n,1)。Tensorflow 采用 1 维表示，本实验将沿用这一约定：
               

<a name="4.4"></a>
### 4.4 Tensorflow 模型实现


Tensorflow 模型是逐层构建的。层的输入维度（上面的 $s_{in}$）会自动为你计算。你指定层的*输出维度*，它决定了下一层的输入维度。第一层的输入维度来自下面 `model.fit` 语句中指定的输入数据大小。
>**注意：** 也可以添加一个输入层来指定第一层的输入维度。例如：  
`tf.keras.Input(shape=(400,)),    #specify input shape`  
我们这里会加上它，以说明模型的尺寸确定过程。

<a name="4.5"></a>
### 4.5 Softmax 的位置
正如课程和可选的 softmax 实验所述，如果在训练时把 softmax 与损失函数归为一组、而不是放在输出层，数值稳定性会更好。这对*构建*模型和*使用*模型都有影响。  
构建时：  
* 最后一个 Dense 层应使用 'linear' 激活函数。这实际上相当于不加激活函数。
* `model.compile` 语句通过包含 `from_logits=True` 来表明这一点。
`loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) `  
* 这不影响目标的形式。在 SparseCategorialCrossentropy 的情况下，目标是期望的数字 0-9。

使用模型时：
* 输出不是概率。如果需要输出概率，请应用 softmax 函数。

<a name="ex02"></a>
### 练习 2

下面，使用 Keras 的 [Sequential 模型](https://keras.io/guides/sequential_model/)和带 ReLU 激活函数的[密集层](https://keras.io/api/layers/core_layers/dense/)来构建上面描述的三层网络。

In [ ]:
# UNQ_C2
# 计分单元：Sequential 模型
tf.random.set_seed(1234) # 为得到一致的结果
model = Sequential(
    [               
        ### 在此处编写代码 ###

        ### 代码编写结束 ###
    ], name = "my_model" 
)

In [ ]:
model.summary()

<details>
  <summary><font size="3" color="darkgreen"><b>预期输出（点击展开）</b></font></summary>
<code>model.summary()</code> 函数会显示模型的实用摘要。注意，除非指定名称，层的名称是自动生成的，可能各不相同。  
    

```
Model: "my_model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
=================================================================
L1 (Dense)                   (None, 25)                10025     
_________________________________________________________________
L2 (Dense)                   (None, 15)                390       
_________________________________________________________________
L3 (Dense)                   (None, 10)                160       
=================================================================
Total params: 10,575
Trainable params: 10,575
Non-trainable params: 0
_________________________________________________________________
```


<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    

```python
tf.random.set_seed(1234)
model = Sequential(
    [               
        ### START CODE HERE ### 
        tf.keras.Input(shape=(400,)),     # @REPLACE 
        Dense(25, activation='relu', name = "L1"), # @REPLACE 
        Dense(15, activation='relu',  name = "L2"), # @REPLACE  
        Dense(10, activation='linear', name = "L3"),  # @REPLACE 
        ### END CODE HERE ### 
    ], name = "my_model" 
)
```
 

In [ ]:
# 单元测试开始
test_model(model, 10, 400)
# 单元测试结束

摘要中显示的参数数量对应下面所示的权重和偏置数组中的元素个数。

让我们进一步检查权重，验证 tensorflow 生成的维度与我们上面计算的一致。

In [ ]:
[layer1, layer2, layer3] = model.layers

In [ ]:
#### 检查权重的形状
W1,b1 = layer1.get_weights()
W2,b2 = layer2.get_weights()
W3,b3 = layer3.get_weights()
print(f"W1 shape = {W1.shape}, b1 shape = {b1.shape}")
print(f"W2 shape = {W2.shape}, b2 shape = {b2.shape}")
print(f"W3 shape = {W3.shape}, b3 shape = {b3.shape}")

**预期输出**

```
W1 shape = (400, 25), b1 shape = (25,)  
W2 shape = (25, 15), b2 shape = (15,)  
W3 shape = (15, 1), b3 shape = (10,)
```


下面的代码：
* 定义损失函数 `SparseCategoricalCrossentropy`，并通过加上 `from_logits=True` 表明 softmax 应包含在损失计算中
* 定义优化器。一个常用的选择是自适应矩估计（Adam），课上已介绍过。

In [ ]:
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
)

history = model.fit(
    X,y,
    epochs=40
)

#### 轮数与批大小
在上面的 `compile` 语句中，`epochs` 设为 100。这表示整个数据集在训练中应被使用 100 次。训练期间，你会看到描述训练进度的输出，形如：

```
Epoch 1/100
157/157 [==============================] - 0s 1ms/step - loss: 2.2770
```

第一行 `Epoch 1/100` 描述模型当前正在运行第几个轮次。为提高效率，训练数据集被拆分为若干“批”。Tensorflow 中一批的默认大小是 32。我们的数据集有 5000 个样本，约 157 批。第 2 行中的 `157/157 [====` 表示已经执行到哪一批。

#### 损失（代价）
在课程 1 中，我们学会了通过监控代价来跟踪梯度下降的进展。理想情况下，随着算法迭代次数增加，代价会下降。Tensorflow 把代价称为 `loss`。上面，在 `model.fit` 执行时，你看到每个轮次都显示损失。[.fit](https://www.tensorflow.org/api_docs/python/tf/keras/Model) 方法返回包括损失在内的多种指标，它们被保存在上面的 `history` 变量中。可以用它像下面这样把损失画成图来查看。

In [ ]:
plot_loss_tf(history)

#### 预测
要做预测，请使用 Keras 的 `predict`。下面，X[1015] 包含一张数字 2 的图像。

In [ ]:
image_of_two = X[1015]
display_digit(image_of_two)

prediction = model.predict(image_of_two.reshape(1,400))  # 预测

print(f" predicting a Two: \n{prediction}")
print(f" Largest Prediction index: {np.argmax(prediction)}")

最大的输出是 prediction[2]，表明预测数字是 '2'。如果问题只需要做出选择，这就够了。可以用 NumPy 的 [argmax](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html) 把它选出来。如果问题需要概率，则需要 softmax：

In [ ]:
prediction_p = tf.nn.softmax(prediction)

print(f" predicting a Two. Probability vector: \n{prediction_p}")
print(f"Total of predictions: {np.sum(prediction_p):0.3f}")

要返回一个表示预测目标的整数，你需要最大概率的索引。这可以用 NumPy 的 [argmax](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html) 函数完成。

In [ ]:
yhat = np.argmax(prediction_p)

print(f"np.argmax(prediction_p): {yhat}")

让我们把随机抽取的 64 个数字的预测结果与标签做比较。这需要运行一会儿。

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# 你不需要修改本单元格中的任何内容

m, n = X.shape

fig, axes = plt.subplots(8,8, figsize=(5,5))
fig.tight_layout(pad=0.13,rect=[0, 0.03, 1, 0.91]) #[左, 下, 右, 上]
widgvis(fig)
for i,ax in enumerate(axes.flat):
    # 随机选取索引
    random_index = np.random.randint(m)
    
    # 选取与随机索引对应的行，并
    # 重塑图像
    X_random_reshaped = X[random_index].reshape((20,20)).T
    
    # 显示图像
    ax.imshow(X_random_reshaped, cmap='gray')
    
    # 使用神经网络做预测
    prediction = model.predict(X[random_index].reshape(1,400))
    prediction_p = tf.nn.softmax(prediction)
    yhat = np.argmax(prediction_p)
    
    # 在图像上方显示标签
    ax.set_title(f"{y[random_index,0]},{yhat}",fontsize=10)
    ax.set_axis_off()
fig.suptitle("Label, yhat", fontsize=14)
plt.show()

让我们看看一些错误。
>注意：增加训练轮数可以消除这个数据集上的错误。

In [ ]:
print( f"{display_errors(model,X,y)} errors out of {len(X)} images")

### 恭喜！
你已经成功构建并使用神经网络完成了多类别分类。